# 02 — Data Cleaning & Alignment

This notebook performs the Phase 2 workflow:
1. **Load** raw datasets from Phase 1.
2. **Clean** individual datasets (parsing dates, checking duplicates, sorting chronologically).
3. **Align** all market datasets to their strict common trading days (inner join).
4. **Save** results to `data/processed/cleaned/` and `data/processed/aligned/`.

## 1. Setup

In [1]:
import sys
import logging
from pathlib import Path
import pandas as pd

# Add project root to path for src imports
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import clean_market_data, clean_event_data, align_market_datasets

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger("Phase2")

# Paths
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "processed" / "cleaned"
ALIGNED_DIR = PROJECT_ROOT / "data" / "processed" / "aligned"

CLEANED_DIR.mkdir(parents=True, exist_ok=True)
ALIGNED_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load Raw Data

In [2]:
# Define raw dataset paths based on verified Phase 1 outputs
DATASETS = {
    "nifty50": RAW_DIR / "market" / "nifty50.csv",
    "india_vix": RAW_DIR / "market" / "india_vix.csv",
    "nifty_it": RAW_DIR / "sector_indices" / "nifty_it.csv",
    "nifty_bank": RAW_DIR / "sector_indices" / "nifty_bank.csv",
    "nifty_auto": RAW_DIR / "sector_indices" / "nifty_auto.csv",
    "nifty_pharma": RAW_DIR / "sector_indices" / "nifty_pharma.csv",
    "nifty_fmcg": RAW_DIR / "sector_indices" / "nifty_fmcg.csv",
    "nifty_metal": RAW_DIR / "sector_indices" / "nifty_metal.csv",
}

raw_market_dfs = {}
for name, path in DATASETS.items():
    if path.exists():
        raw_market_dfs[name] = pd.read_csv(path, parse_dates=["Date"], index_col="Date")
        logger.info(f"Loaded {name} ({len(raw_market_dfs[name])} rows)")
    else:
        logger.error(f"Raw file not found: {path}")

# Load Events
EVENT_FILE = RAW_DIR / "Events" / "major_events_2019_2026.csv"
if EVENT_FILE.exists():
    raw_events_df = pd.read_csv(EVENT_FILE)
    logger.info(f"Loaded events ({len(raw_events_df)} rows)")
else:
    logger.error(f"Event file not found: {EVENT_FILE}")

2026-09-20 23:28:12  INFO      Loaded nifty50 (1903 rows)
2026-09-20 23:28:12  INFO      Loaded india_vix (1887 rows)
2026-09-20 23:28:12  INFO      Loaded nifty_it (1903 rows)
2026-09-20 23:28:12  INFO      Loaded nifty_bank (1903 rows)
2026-09-20 23:28:12  INFO      Loaded nifty_auto (1905 rows)
2026-09-20 23:28:12  INFO      Loaded nifty_pharma (1902 rows)
2026-09-20 23:28:12  INFO      Loaded nifty_fmcg (1905 rows)
2026-09-20 23:28:12  INFO      Loaded nifty_metal (1905 rows)
2026-09-20 23:28:12  INFO      Loaded events (135 rows)


## 3. Data Cleaning
Here we enforce correct date types, check for duplicates, and ensure chronological order. Then we save the cleaned individual datasets.

In [3]:
cleaned_market_dfs = {}

for name, df in raw_market_dfs.items():
    cleaned_df = clean_market_data(df, dataset_name=name)
    cleaned_market_dfs[name] = cleaned_df
    
    # Save to data/processed/cleaned/
    out_path = CLEANED_DIR / f"{name}_cleaned.csv"
    cleaned_df.to_csv(out_path)

logger.info("All market datasets cleaned and saved.")

2026-09-20 23:28:12  INFO      [nifty50] Cleaned. Dropped 0 exact dupes, 0 index dupes. Final rows: 1903.
2026-09-20 23:28:12  INFO      [india_vix] Cleaned. Dropped 0 exact dupes, 0 index dupes. Final rows: 1887.
2026-09-20 23:28:12  INFO      [nifty_it] Cleaned. Dropped 0 exact dupes, 0 index dupes. Final rows: 1903.
2026-09-20 23:28:12  INFO      [nifty_bank] Cleaned. Dropped 0 exact dupes, 0 index dupes. Final rows: 1903.
2026-09-20 23:28:12  INFO      [nifty_auto] Cleaned. Dropped 0 exact dupes, 0 index dupes. Final rows: 1905.
2026-09-20 23:28:12  INFO      [nifty_pharma] Cleaned. Dropped 0 exact dupes, 0 index dupes. Final rows: 1902.
2026-09-20 23:28:12  INFO      [nifty_fmcg] Cleaned. Dropped 0 exact dupes, 0 index dupes. Final rows: 1905.
2026-09-20 23:28:12  INFO      [nifty_metal] Cleaned. Dropped 0 exact dupes, 0 index dupes. Final rows: 1905.
2026-09-20 23:28:12  INFO      All market datasets cleaned and saved.


In [4]:
if 'raw_events_df' in locals():
    cleaned_events_df = clean_event_data(raw_events_df)
    
    # Save to data/processed/cleaned/
    out_path = CLEANED_DIR / "major_events_cleaned.csv"
    cleaned_events_df.to_csv(out_path, index=False)
    logger.info("Event dataset cleaned and saved.")

2026-09-20 23:28:12  INFO      [Events] Parsed dates. Total events: 135.
2026-09-20 23:28:12  INFO      Event dataset cleaned and saved.


## 4. Data Alignment
We intersect the trading days of all market datasets to produce a strictly aligned dataset for each instrument. This handles missing observations (like holidays specific to certain indices) without inventing data.

In [5]:
aligned_market_dfs = align_market_datasets(cleaned_market_dfs)

for name, df in aligned_market_dfs.items():
    # Save to data/processed/aligned/
    out_path = ALIGNED_DIR / f"{name}_aligned.csv"
    df.to_csv(out_path)

logger.info("All aligned datasets saved.")

2026-09-20 23:28:12  INFO      [Alignment] Found 1886 common trading days across 8 datasets.
2026-09-20 23:28:12  INFO      [nifty50] Aligned from 1903 -> 1886 rows.
2026-09-20 23:28:12  INFO      [india_vix] Aligned from 1887 -> 1886 rows.
2026-09-20 23:28:12  INFO      [nifty_it] Aligned from 1903 -> 1886 rows.
2026-09-20 23:28:12  INFO      [nifty_bank] Aligned from 1903 -> 1886 rows.
2026-09-20 23:28:12  INFO      [nifty_auto] Aligned from 1905 -> 1886 rows.
2026-09-20 23:28:12  INFO      [nifty_pharma] Aligned from 1902 -> 1886 rows.
2026-09-20 23:28:12  INFO      [nifty_fmcg] Aligned from 1905 -> 1886 rows.
2026-09-20 23:28:12  INFO      [nifty_metal] Aligned from 1905 -> 1886 rows.
2026-09-20 23:28:12  INFO      All aligned datasets saved.


## 5. Validation & Summary
Verify that the aligned datasets are uniform.

In [6]:
if aligned_market_dfs:
    base_name = list(aligned_market_dfs.keys())[0]
    base_idx = aligned_market_dfs[base_name].index
    
    all_matched = True
    for name, df in aligned_market_dfs.items():
        if not df.index.equals(base_idx):
            logger.error(f"Alignment mismatch in {name}!")
            all_matched = False
            
    if all_matched:
        logger.info(f"SUCCESS: All {len(aligned_market_dfs)} aligned datasets have exactly {len(base_idx)} trading days.")
        logger.info(f"Start Date: {base_idx[0].date()} | End Date: {base_idx[-1].date()}")
    else:
        logger.error("FAILURE: Index alignment check failed.")

2026-09-20 23:28:12  INFO      SUCCESS: All 8 aligned datasets have exactly 1886 trading days.
2026-09-20 23:28:12  INFO      Start Date: 2019-01-02 | End Date: 2026-09-18
